In [ ]:
# print("123")

In [1]:
!pip install minsearch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!pip install pydantic


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
!pip install google-genai

Traceback (most recent call last):
  File "/home/codespace/.python/current/bin/pip", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/cli/main.py", line 47, in main
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/cli/autocompletion.py", line 12, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/cli/main_parser.py", line 11, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/build_env.py", line 22, in <module>
    from pip._internal.cli.spinners import open_rich_spinner, open_spinner
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/cli/spinners.py", line 11, in <module>
    from pip._vendor.r

In [5]:
!pip install tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [31]:
import os
import json
import re
import pandas as pd
from ingest import load_faq_data
from evaluation_utils import llm_structured, llm_structured_retry
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [8]:
model='gemini-2.5-flash'

In [9]:
documents = load_faq_data()
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [10]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [11]:
class Question(BaseModel):
    questions: list[str]

In [12]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
# user_prompt = json.dumps(doc, indent=4)

In [ ]:
# print(user_prompt)

{
    "id": "ab183bd688",
    "course": "machine-learning-zoomcamp",
    "section": "Miscellaneous",
    "question": "My homework answer doesn't match any of the options",
    "answer": "Common causes, in order of frequency:\n\n1. Wrong column slice or filter \u2014 apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early \u2014 only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions \u2014 pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic \u2014 `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option \u2014 the homework explicitly allows it."
}


In [13]:
user_prompt = "FAQ Record: The course platform is accessible 24/7, and certificates are issued automatically upon passing the final exam."

In [14]:
messages = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(text=user_prompt), 
            types.Part.from_text(text=data_gen_instructions)
        ]
    )
]

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=2000,
    response_mime_type="application/json",  # Forces Gemini to output strict JSON
    response_schema=Question                # Enforces your structural blueprint
)

response = client.models.generate_content(
    model=model,
    contents=messages,
    config=config
)

In [15]:
raw_text = response.text.strip()

In [16]:
try:
    if response.parsed is not None:
        questions_list = response.parsed.questions
    else:
        questions_list = json.loads(raw_text).get("questions", [])
except json.JSONDecodeError:
    # If it is still truncated or malformed, attempt to clean/patch it manually
    print("Received malformed JSON string from API. Attempting repair...")
    
    # 1. Strip markdown code fence wrappers if Gemini accidentally included them
    clean_text = re.sub(r"^```json\s*|\s*```$", "", raw_text, flags=re.MULTILINE).strip()
    
    # 2. Basic auto-closure check if it got cut off near the end
    if not clean_text.endswith("]}"):
        if clean_text.endswith('"'): clean_text += "]}"
        elif not clean_text.endswith(']'): clean_text += '"]}'
    
    try:
        questions_list = json.loads(clean_text).get("questions", [])
    except Exception as e:
        # Final safety net fallback
        print(f"Could not repair JSON. Raw response was: \n{raw_text}")
        questions_list = []

In [17]:
print("Final Output Questions:", questions_list)

Final Output Questions: ['Is the course platform available 24 hours a day?', "Are there any specific times I won't be able to access the course content?", "What's the process for getting my certificate after I complete the course?", 'Do I need to pass a final exam to receive my certificate?', 'Will my certificate be sent to me automatically, or do I need to request it?']


In [18]:
doc

{'id': 'ab183bd688',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': "My homework answer doesn't match any of the options",
 'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early — only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions — pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic — `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option — the homework explicitly allows it."}

In [19]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    Question
)

print(result.questions)

['Can I access the course platform at any time of day?', "What's the process for getting my certificate after I finish the course?", 'Do I need to manually request my certificate, or is it automatically provided?', 'Is there a specific deadline or time window for accessing the course materials?', 'What do I need to do to qualify for the course completion certificate?']


In [ ]:
# # Extract token usage from metadata
# usage = response.usage_metadata
# prompt_tokens = usage.prompt_token_count
# candidate_tokens = usage.candidates_token_count

# # Print individual metrics
# print(f"Prompt (Input) Tokens: {prompt_tokens}")
# print(f"Candidates (Output) Tokens: {candidate_tokens}")
# print(f"Total Tokens Used: {usage.total_token_count}")

Prompt (Input) Tokens: 115
Candidates (Output) Tokens: 83
Total Tokens Used: 1426


In [26]:
usage = response.usage_metadata

In [24]:
def calc_price(usage):
    usage = response.usage_metadata
    prompt_tokens = usage.prompt_token_count
    candidate_tokens = usage.candidates_token_count
    
    input_price_per_million = 0.75
    output_price_per_million = 4.50

    input_cost = (prompt_tokens / 1_000_000) * input_price_per_million
    output_cost = (candidate_tokens / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

In [25]:
def calc_total_price(usages):
    total_cost = 0.0

    for usage in usages:
        cost = calc_price(usage)
        total_cost = total_cost + cost["total_cost"]

    return total_cost

In [27]:
calc_price(usage)

{'input_cost': 8.625000000000001e-05,
 'output_cost': 0.00037349999999999997,
 'total_cost': 0.00045975}

In [28]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I access the course platform at any time of day?',
  'document': 'ab183bd688'},
 {'question': "What's the process for getting my certificate after I finish the course?",
  'document': 'ab183bd688'},
 {'question': 'Do I need to manually request my certificate, or is it automatically provided?',
  'document': 'ab183bd688'},
 {'question': 'Is there a specific deadline or time window for accessing the course materials?',
  'document': 'ab183bd688'},
 {'question': 'What do I need to do to qualify for the course completion certificate?',
  'document': 'ab183bd688'}]

In [30]:
pd.DataFrame(records)

,question,document
0,Can I access the course platform at any time o...,ab183bd688
1,What's the process for getting my certificate ...,ab183bd688
2,"Do I need to manually request my certificate, ...",ab183bd688
3,Is there a specific deadline or time window fo...,ab183bd688
4,What do I need to do to qualify for the course...,ab183bd688


In [33]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Question
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [34]:
generate_ground_truth(doc)

([{'question': "I'm getting different results for my homework. Does the order of operations matter when I'm filtering or selecting columns?",
   'document': 'ab183bd688'},
  {'question': 'My answer is off. Could it be because I applied a log transformation at the wrong time?',
   'document': 'ab183bd688'},
  {'question': 'Should I round intermediate calculations or only the final answer for the homework problems?',
   'document': 'ab183bd688'},
  {'question': 'Is it possible that my Python or library versions are causing my homework answers to be different?',
   'document': 'ab183bd688'},
  {'question': "Why might my train-test split produce different results than what's expected in the homework?",
   'document': 'ab183bd688'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=105,
   prompt_token_count=343,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=343
     ),
   ],
   thoughts_token_count=820,
